In [1]:
import os
import json
import joblib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from lightgbm import LGBMRegressor

## 2) Load data

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/netload/f10_netload_nasa_clean.csv")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

print(df.shape)
df.head()

(14784, 6)


,Timestamp,MW_raw,NetLoad_MW,ALLSKY_SFC_SW_DWN,T2M,WS10M
0,2025-03-31 15:45:00,1.356,-1.356,425.48,31.07,3.42
1,2025-03-31 16:00:00,0.353,-0.353,194.80,30.73,3.49
2,2025-03-31 16:15:00,-0.509,0.509,194.80,30.73,3.49
3,2025-03-31 16:30:00,-1.902,1.902,194.80,30.73,3.49
4,2025-03-31 16:45:00,-2.148,2.148,194.80,30.73,3.49


## 3) Match the hybrid forecasting setup

In [3]:
L_INPUT = 192
H_HORIZON = 96
ISSUE_TIME = "23:45"

TARGET_COL = "NetLoad_MW"
EXOG_COLS = ["ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]
FEATURE_COLS = [TARGET_COL] + EXOG_COLS

print("Target:", TARGET_COL)
print("Features:", FEATURE_COLS)
print("Input window:", L_INPUT)
print("Forecast horizon:", H_HORIZON)
print("Issue time:", ISSUE_TIME)

Target: NetLoad_MW
Features: ['NetLoad_MW', 'ALLSKY_SFC_SW_DWN', 'T2M', 'WS10M']
Input window: 192
Forecast horizon: 96
Issue time: 23:45


## 4) Create supervised samples

In [4]:
def create_direct_multistep_samples(
    df,
    feature_cols,
    target_col,
    input_len=192,
    horizon=96,
    issue_time="23:45"
):
    X_list = []
    y_list = []
    issue_ts_list = []
    target_start_list = []

    values_feat = df[feature_cols].values
    values_target = df[target_col].values
    timestamps = df["Timestamp"].values

    for i in range(input_len, len(df) - horizon + 1):
        ts = pd.Timestamp(timestamps[i - 1])

        if ts.strftime("%H:%M") != issue_time:
            continue

        x_window = values_feat[i - input_len:i]
        y_future = values_target[i:i + horizon]

        X_list.append(x_window.reshape(-1))
        y_list.append(y_future)
        issue_ts_list.append(ts)
        target_start_list.append(pd.Timestamp(timestamps[i]))

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    meta = pd.DataFrame({
        "issue_time": issue_ts_list,
        "target_start": target_start_list
    })

    return X, y, meta

In [5]:
X_all, y_all, meta_all = create_direct_multistep_samples(
    df=df,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    input_len=L_INPUT,
    horizon=H_HORIZON,
    issue_time=ISSUE_TIME
)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print(meta_all.head())
print(meta_all.tail())

X_all shape: (151, 768)
y_all shape: (151, 96)
           issue_time target_start
0 2025-04-02 23:45:00   2025-04-03
1 2025-04-03 23:45:00   2025-04-04
2 2025-04-04 23:45:00   2025-04-05
3 2025-04-05 23:45:00   2025-04-06
4 2025-04-06 23:45:00   2025-04-07
             issue_time target_start
146 2025-08-26 23:45:00   2025-08-27
147 2025-08-27 23:45:00   2025-08-28
148 2025-08-28 23:45:00   2025-08-29
149 2025-08-29 23:45:00   2025-08-30
150 2025-08-30 23:45:00   2025-08-31


## 5) Train / validation / test split

In [6]:
TRAIN_END = pd.Timestamp("2025-06-30 23:45:00")
VAL_END   = pd.Timestamp("2025-07-31 23:45:00")
TEST_END  = pd.Timestamp("2025-08-30 23:45:00")

train_mask = meta_all["issue_time"] <= TRAIN_END
val_mask   = (meta_all["issue_time"] > TRAIN_END) & (meta_all["issue_time"] <= VAL_END)
test_mask  = (meta_all["issue_time"] > VAL_END) & (meta_all["issue_time"] <= TEST_END)

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val     = X_all[val_mask], y_all[val_mask]
X_test, y_test   = X_all[test_mask], y_all[test_mask]

meta_train = meta_all.loc[train_mask].reset_index(drop=True)
meta_val   = meta_all.loc[val_mask].reset_index(drop=True)
meta_test  = meta_all.loc[test_mask].reset_index(drop=True)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

print("\nTrain range:")
print(meta_train.head(1))
print(meta_train.tail(1))

print("\nVal range:")
print(meta_val.head(1))
print(meta_val.tail(1))

print("\nTest range:")
print(meta_test.head(1))
print(meta_test.tail(1))

Train: (90, 768) (90, 96)
Val  : (31, 768) (31, 96)
Test : (30, 768) (30, 96)

Train range:
           issue_time target_start
0 2025-04-02 23:45:00   2025-04-03
            issue_time target_start
89 2025-06-30 23:45:00   2025-07-01

Val range:
           issue_time target_start
0 2025-07-01 23:45:00   2025-07-02
            issue_time target_start
30 2025-07-31 23:45:00   2025-08-01

Test range:
           issue_time target_start
0 2025-08-01 23:45:00   2025-08-02
            issue_time target_start
29 2025-08-30 23:45:00   2025-08-31


## model training import block:

In [7]:
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

## train with this lightgbm baseline

In [8]:
base_lgbm = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1
)

model = MultiOutputRegressor(base_lgbm, n_jobs=-1)
model.fit(X_train, y_train)

print("LightGBM baseline trained.")

LightGBM baseline trained.


## 1) Predict on train / val / test


In [9]:
pred_train = model.predict(X_train)
pred_val   = model.predict(X_val)
pred_test  = model.predict(X_test)

print("Train pred shape:", pred_train.shape)
print("Val pred shape  :", pred_val.shape)
print("Test pred shape :", pred_test.shape)

Train pred shape: (90, 96)
Val pred shape  : (31, 96)
Test pred shape : (30, 96)


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def r2_percent_display(a, b):
    r2_val = r2_score(a.reshape(-1), b.reshape(-1)) * 100
    return f"{r2_val:.2f}%"

results_df_lgbm = pd.DataFrame([
    [
        "LightGBM Baseline", "Train",
        round(mae(y_train, pred_train), 4),
        round(rmse(y_train, pred_train), 4),
        r2_percent_display(y_train, pred_train)
    ],
    [
        "LightGBM Baseline", "Val",
        round(mae(y_val, pred_val), 4),
        round(rmse(y_val, pred_val), 4),
        r2_percent_display(y_val, pred_val)
    ],
    [
        "LightGBM Baseline", "Test",
        round(mae(y_test, pred_test), 4),
        round(rmse(y_test, pred_test), 4),
        r2_percent_display(y_test, pred_test)
    ],
], columns=["Model", "Split", "MAE_MW", "RMSE_MW", "R2_Percentage"])

results_df_lgbm

,Model,Split,MAE_MW,RMSE_MW,R2_Percentage
0,LightGBM Baseline,Train,0.1233,0.2654,99.75%
1,LightGBM Baseline,Val,0.8441,1.3587,93.65%
2,LightGBM Baseline,Test,1.1257,1.7498,89.75%
